# 06 — Advanced Forecasting Models

> **Objective:** Build multiple forecasting models (ARIMA, SARIMA, LightGBM, GradientBoosting) and create ensemble predictions to improve upon the Prophet baseline.

---

## 1. Setup and Load Data

Import libraries, load raw CSV, and prepare daily aggregated series.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import lightgbm as lgb

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_raw_weather

reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

df = load_raw_weather(project_root)
df = df.dropna(subset=["last_updated", "temperature_celsius"]).copy()

print("Shape after dropna:", df.shape)
print("Time range:", df["last_updated"].min(), "->", df["last_updated"].max())

## 2. Daily Aggregation

Aggregate to daily mean temperature (same as Prophet baseline).

In [ ]:
df["date"] = df["last_updated"].dt.normalize()
daily = df.groupby("date", as_index=False)["temperature_celsius"].mean()
daily.columns = ["ds", "y"]
daily["ds"] = pd.to_datetime(daily["ds"])
daily = daily.sort_values("ds").reset_index(drop=True)

print("Daily series shape:", daily.shape)
print("Date range:", daily["ds"].min(), "->", daily["ds"].max())
display(daily.head())

## 3. Train/Test Split

Temporal holdout: last 30 days for testing.

In [ ]:
cutoff = daily["ds"].max() - pd.Timedelta(days=30)
train = daily[daily["ds"] <= cutoff].copy()
test = daily[daily["ds"] > cutoff].copy()

print(f"TRAIN: n={len(train)}, {train['ds'].min()} -> {train['ds'].max()}")
print(f"TEST:  n={len(test)}, {test['ds'].min()} -> {test['ds'].max()}")

## 4. Stationarity Analysis

Augmented Dickey-Fuller test and ACF/PACF plots.

In [ ]:
series = train.set_index("ds")["y"]

adf_result = adfuller(series, autolag="AIC")
print("ADF Test Results:")
print(f"  Test Statistic: {adf_result[0]:.4f}")
print(f"  p-value: {adf_result[1]:.4f}")
print(f"  Lags Used: {adf_result[2]}")
print(f"  Critical Values:")
for key, val in adf_result[4].items():
    print(f"    {key}: {val:.4f}")

if adf_result[1] < 0.05:
    print("\n=> Series is stationary (reject H0)")
else:
    print("\n=> Series is non-stationary (fail to reject H0)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(series, lags=40, ax=axes[0], title="Autocorrelation Function (ACF)")
plot_pacf(series, lags=40, ax=axes[1], title="Partial Autocorrelation Function (PACF)")

plt.tight_layout()
out_path = reports_dir / "forecast_acf_pacf.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out_path)

## 5. ARIMA Model

Fit ARIMA(5,1,0) based on PACF cutoff analysis.

In [ ]:
arima_model = ARIMA(series, order=(5, 1, 0))
arima_fit = arima_model.fit()

print(arima_fit.summary())

In [ ]:
arima_pred = arima_fit.forecast(steps=len(test))
arima_pred = arima_pred.values

print("ARIMA predictions shape:", arima_pred.shape)

## 6. SARIMA Model

SARIMA(1,1,1)(1,1,1,7) with weekly seasonality.

In [ ]:
sarima_model = SARIMAX(
    series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarima_fit = sarima_model.fit(disp=False)

print(sarima_fit.summary())

In [ ]:
sarima_pred = sarima_fit.forecast(steps=len(test))
sarima_pred = sarima_pred.values

print("SARIMA predictions shape:", sarima_pred.shape)

## 7. Feature Engineering for ML Models

Create lag features, rolling statistics, and calendar features.

In [ ]:
def create_features(df):
    """Create time series features for ML models."""
    data = df.copy()
    data = data.set_index("ds")
    
    # Lag features
    for lag in [1, 2, 3, 7, 14, 21]:
        data[f"lag_{lag}"] = data["y"].shift(lag)
    
    # Rolling statistics
    for window in [7, 14, 30]:
        data[f"rolling_mean_{window}"] = data["y"].shift(1).rolling(window=window).mean()
        data[f"rolling_std_{window}"] = data["y"].shift(1).rolling(window=window).std()
    
    # Calendar features
    data["dayofweek"] = data.index.dayofweek
    data["month"] = data.index.month
    data["dayofyear"] = data.index.dayofyear
    data["weekofyear"] = data.index.isocalendar().week.astype(int)
    
    # Cyclical encoding for month
    data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)
    
    # Cyclical encoding for day of week
    data["dow_sin"] = np.sin(2 * np.pi * data["dayofweek"] / 7)
    data["dow_cos"] = np.cos(2 * np.pi * data["dayofweek"] / 7)
    
    return data

ml_data = create_features(daily)
ml_data = ml_data.dropna()

print("ML data shape:", ml_data.shape)
print("Features:", ml_data.columns.tolist())

In [ ]:
# Split ML data
ml_train = ml_data[ml_data.index <= cutoff].copy()
ml_test = ml_data[ml_data.index > cutoff].copy()

feature_cols = [c for c in ml_data.columns if c != "y"]

X_train = ml_train[feature_cols]
y_train = ml_train["y"]
X_test = ml_test[feature_cols]
y_test = ml_test["y"]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 8. LightGBM Model

Train LightGBM with early stopping.

In [ ]:
# Carve a chronological validation slice from the TAIL of the training window
# for early stopping (no shuffle). The held-out test window is left untouched
# and scored exactly once, avoiding evaluation leakage (issue #20).
val_size = 30
X_tr, X_val = X_train.iloc[:-val_size], X_train.iloc[-val_size:]
y_tr, y_val = y_train.iloc[:-val_size], y_train.iloc[-val_size:]

lgb_train = lgb.Dataset(X_tr, label=y_tr)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "seed": 42,
}

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "valid"],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print(
    f"Validation tail (early stopping): n={val_size}, "
    f"{X_val.index.min().date()} -> {X_val.index.max().date()}"
)
print(f"Best iteration: {lgb_model.best_iteration}")

In [ ]:
lgb_pred = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
print("LightGBM predictions shape:", lgb_pred.shape)

## 9. GradientBoosting Model

Sklearn GradientBoostingRegressor as alternative ensemble method.

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.8,
    random_state=42,
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)
print("GradientBoosting predictions shape:", gb_pred.shape)

## 10. Ensemble Methods

Simple average and weighted average (inverse RMSE weighting).

In [ ]:
def calc_metrics(y_true, y_pred, model_name):
    """Calculate RMSE, MAE, and MAPE."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return {"Model": model_name, "RMSE": rmse, "MAE": mae, "MAPE": mape}

y_true = test["y"].values

# Individual model metrics
results = []
results.append(calc_metrics(y_true, arima_pred, "ARIMA(5,1,0)"))
results.append(calc_metrics(y_true, sarima_pred, "SARIMA(1,1,1)(1,1,1,7)"))
results.append(calc_metrics(y_test.values, lgb_pred, "LightGBM"))
results.append(calc_metrics(y_test.values, gb_pred, "GradientBoosting"))

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Simple Average Ensemble
ensemble_simple = (arima_pred + sarima_pred + lgb_pred + gb_pred) / 4
results.append(calc_metrics(y_true, ensemble_simple, "Ensemble (Simple Avg)"))

# Weighted Ensemble (inverse RMSE weighting)
rmse_arima = results_df[results_df["Model"] == "ARIMA(5,1,0)"]["RMSE"].values[0]
rmse_sarima = results_df[results_df["Model"] == "SARIMA(1,1,1)(1,1,1,7)"]["RMSE"].values[0]
rmse_lgb = results_df[results_df["Model"] == "LightGBM"]["RMSE"].values[0]
rmse_gb = results_df[results_df["Model"] == "GradientBoosting"]["RMSE"].values[0]

weights = 1 / np.array([rmse_arima, rmse_sarima, rmse_lgb, rmse_gb])
weights = weights / weights.sum()

print("Ensemble weights (inverse RMSE):")
print(f"  ARIMA: {weights[0]:.3f}")
print(f"  SARIMA: {weights[1]:.3f}")
print(f"  LightGBM: {weights[2]:.3f}")
print(f"  GradientBoosting: {weights[3]:.3f}")

ensemble_weighted = (
    weights[0] * arima_pred +
    weights[1] * sarima_pred +
    weights[2] * lgb_pred +
    weights[3] * gb_pred
)
results.append(calc_metrics(y_true, ensemble_weighted, "Ensemble (Weighted)"))

## 11. Model Comparison

Compare all models including Prophet baseline.

In [ ]:
# Add Prophet baseline for comparison
results.append({"Model": "Prophet (Baseline)", "RMSE": 0.77, "MAE": 0.69, "MAPE": 3.95})

comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values("RMSE").reset_index(drop=True)

print("\n=== Model Comparison ===")
display(comparison_df.round(4))

In [ ]:
# Visualization: Model Comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = sns.color_palette("viridis", n_colors=len(comparison_df))

# RMSE
ax = axes[0]
bars = ax.barh(comparison_df["Model"], comparison_df["RMSE"], color=colors)
ax.set_xlabel("RMSE (°C)")
ax.set_title("RMSE Comparison")
ax.invert_yaxis()
for bar, val in zip(bars, comparison_df["RMSE"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center")

# MAE
ax = axes[1]
bars = ax.barh(comparison_df["Model"], comparison_df["MAE"], color=colors)
ax.set_xlabel("MAE (°C)")
ax.set_title("MAE Comparison")
ax.invert_yaxis()
for bar, val in zip(bars, comparison_df["MAE"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f"{val:.3f}", va="center")

# MAPE
ax = axes[2]
bars = ax.barh(comparison_df["Model"], comparison_df["MAPE"], color=colors)
ax.set_xlabel("MAPE (%)")
ax.set_title("MAPE Comparison")
ax.invert_yaxis()
for bar, val in zip(bars, comparison_df["MAPE"]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, f"{val:.2f}%", va="center")

plt.tight_layout()
out_path = reports_dir / "forecast_model_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out_path)

In [ ]:
# Visualization: Predictions Comparison
fig, ax = plt.subplots(figsize=(14, 6))

test_dates = test["ds"].values

ax.plot(test_dates, y_true, "ko-", label="Actual", linewidth=2, markersize=6)
ax.plot(test_dates, arima_pred, "--", label="ARIMA", alpha=0.7)
ax.plot(test_dates, sarima_pred, "--", label="SARIMA", alpha=0.7)
ax.plot(test_dates, lgb_pred, "--", label="LightGBM", alpha=0.7)
ax.plot(test_dates, gb_pred, "--", label="GradientBoosting", alpha=0.7)
ax.plot(test_dates, ensemble_weighted, "r-", label="Ensemble (Weighted)", linewidth=2)

ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.set_title("30-Day Forecast: Model Predictions vs Actual")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
out_path = reports_dir / "forecast_predictions_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", out_path)

## 12. Summary

**Models Implemented:**
- ARIMA(5,1,0): Classical statistical model
- SARIMA(1,1,1)(1,1,1,7): Seasonal ARIMA with weekly periodicity
- LightGBM: Gradient boosting with lag and rolling features
- GradientBoosting: Sklearn ensemble regressor
- Simple Average Ensemble: Equal weighting
- Weighted Ensemble: Inverse RMSE weighting

**Key Findings:**
- Feature engineering (lags, rolling stats, calendar features) helps ML models
- Ensemble methods combine strengths of diverse model families
- Weighted ensemble typically outperforms simple average

**Outputs:**
- `reports/forecast_acf_pacf.png`
- `reports/forecast_model_comparison.png`
- `reports/forecast_predictions_comparison.png`